In [1]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Create a dedicated directory to save your testing logs and images if it doesn't exist
LOG_DIR = '/content/drive/MyDrive/driving_dataset/Logs/'
os.makedirs(LOG_DIR, exist_ok=True)
print(f"Logs and verification samples will be saved to: {LOG_DIR}")

Mounted at /content/drive
Logs and verification samples will be saved to: /content/drive/MyDrive/driving_dataset/Logs/


In [2]:
import numpy as np
import pandas as pd
import tensorflow as tf
import cv2
import time
from datetime import datetime
import matplotlib.pyplot as plt
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode, b64encode

In [3]:
MODEL_PATH = '/content/drive/MyDrive/driving_dataset/stage1/best_resnet50_stage1.keras'

print("Loading trained model...")
model = tf.keras.models.load_model(MODEL_PATH)
print("Model loaded successfully!")

Loading trained model...
Model loaded successfully!


In [4]:
# Define target parameters matching training configuration
IMG_SIZE = (224, 224)
CLASS_NAMES = ['both_hands', 'no_hands', 'one_hand']

# Initialize an empty list to keep track of our testing session logs
session_logs = []

In [5]:
# JavaScript helper to handle the browser webcam stream
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
    }

    function createDom() {
       if (div !== null) return;

       div = document.createElement('div');
       div.style.border = '2px solid black';
       div.style.padding = '10px';
       div.style.width = '480px';
       div.style.borderRadius = '10px';
       div.style.background = '#f0f0f0';

       labelElement = document.createElement('div');
       labelElement.style.fontSize = '16px';
       labelElement.style.fontWeight = 'bold';
       labelElement.style.marginBottom = '10px';
       labelElement.innerText = "Camera Feed Initializing...";
       div.appendChild(labelElement);

       video = document.createElement('video');
       video.style.display = 'block';
       video.width = 440;
       video.height = 330;
       video.autoplay = true;
       div.appendChild(video);

       imgElement = document.createElement('img');
       imgElement.style.position = 'absolute';
       imgElement.style.top = '0px';
       imgElement.style.left = '0px';

       var button = document.createElement('button');
       button.innerText = "Stop Testing Feed";
       button.style.marginTop = '10px';
       button.style.padding = '5px 10px';
       button.style.cursor = 'pointer';
       button.onclick = function() {
          shutdown = true;
          if (pendingResolve) {
             pendingResolve({img: null, shutdown: true});
          }
       };
       div.appendChild(button);

       document.body.appendChild(div);
    }

    async function initCamera() {
       createDom();
       stream = await navigator.mediaDevices.getUserMedia({video: true});
       video.srcObject = stream;
       await video.play();

       captureCanvas = document.createElement('canvas');
       captureCanvas.width = video.videoWidth;
       captureCanvas.height = video.videoHeight;
    }

    async function captureFrame(statusText) {
       if (shutdown) {
          removeDom();
          return {img: null, shutdown: true};
       }

       labelElement.innerText = statusText;

       // Wait a brief moment to ensure frame changes
       await new Promise(resolve => setTimeout(resolve, 100));

       var context = captureCanvas.getContext('2d');
       context.drawImage(video, 0, 0, video.videoWidth, video.videoHeight);
       var imgData = captureCanvas.toDataURL('image/jpeg', 0.8);

       return {img: imgData, shutdown: false};
    }

    window.initCamera = initCamera;
    window.captureFrame = captureFrame;
    ''')
  display(js)

# Helper function to convert JavaScript base64 image back to a OpenCV/NumPy BGR image
def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=cv2.IMREAD_COLOR)
  return img

In [11]:
import ipywidgets as widgets
import time

# Configurable delay (in seconds) to give you time to switch positions
PAUSE_DURATION = 4

# Dropdown to select what posture you are simulating right now
simulated_state_widget = widgets.Dropdown(
    options=['no_hands', 'one_hand', 'both_hands'],
    value='one_hand',
    description='Simulating:',
)
display(simulated_state_widget)

# Start the web camera
video_stream()
eval_js('initCamera()')

print(f"Camera started. You have a {PAUSE_DURATION}-second delay between each snapshot to adjust your posture.")
print("Click 'Stop Testing Feed' below the video to finish the session.")

# Reset local log for this specific run
run_logs = []

while True:
    current_target = simulated_state_widget.value
    status_text = f"Simulating: {current_target.upper()} | Capturing frame now..."

    # 1. Capture a frame from the JavaScript side
    js_reply = eval_js(f'captureFrame("{status_text}")')
    if js_reply is None or js_reply.get('shutdown', False):
        print("\nTesting session ended by user.")
        break

    # Process image matrix
    img = js_to_image(js_reply['img'])

    # Preprocess matching your dataset structure
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized_img = cv2.resize(rgb_img, IMG_SIZE)
    input_tensor = np.expand_dims(resized_img, axis=0).astype(np.float32)

    # Core inference execution
    predictions = model.predict(input_tensor, verbose=0)
    score = tf.nn.softmax(predictions[0])
    class_id = np.argmax(score)
    pred_label = CLASS_NAMES[class_id]
    confidence = float(np.max(score))

    # Save log details
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    is_correct = (pred_label == current_target)

    record = {
        "Timestamp": timestamp,
        "Simulated_Ground_Truth": current_target,
        "Model_Prediction": pred_label,
        "Confidence": confidence,
        "Is_Correct": is_correct
    }
    run_logs.append(record)
    session_logs.append(record)

    # Print real-time diagnostic output
    print(f"[{timestamp}] Expected: {current_target:10} | Predicted: {pred_label:10} ({confidence*100:6.2f}%) -> Match: {is_correct}")

    # 2. Countdown sequence to give you a margin to adjust
    print("Preparing for next snapshot... ", end="")
    for i in range(PAUSE_DURATION, 0, -1):
        # Update the video banner dynamically with the remaining countdown time
        countdown_text = f"HOLD POSTURE! Next frame in {i} seconds..."
        eval_js(f'captureFrame("{countdown_text}")')

        print(f"{i}.. ", end="", flush=True)
        time.sleep(1)
    print("Capture!\n" + "-"*60)

Dropdown(description='Simulating:', index=1, options=('no_hands', 'one_hand', 'both_hands'), value='one_hand')

<IPython.core.display.Javascript object>

Camera started. You have a 4-second delay between each snapshot to adjust your posture.
Click 'Stop Testing Feed' below the video to finish the session.
[2026-07-05 10:44:27] Expected: one_hand   | Predicted: one_hand   ( 39.92%) -> Match: True
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 10:44:33] Expected: one_hand   | Predicted: one_hand   ( 48.29%) -> Match: True
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 10:44:39] Expected: one_hand   | Predicted: no_hands   ( 45.03%) -> Match: False
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 10:44:45] Expected: one_hand   | Predicted: one_hand   ( 56.78%) -> Match: True
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-0

## Testing EfficientNET

In [12]:
MODEL_PATH_2 = '/content/drive/MyDrive/driving_dataset/best_stage1_model_v2.keras'

print("Loading trained model...")
model = tf.keras.models.load_model(MODEL_PATH_2)
print("Model loaded successfully!")

Loading trained model...
Model loaded successfully!


In [13]:
IMG_SIZE = (224, 224)
CLASS_NAMES = ['both_hands', 'no_hands', 'one_hand']

# Initialize an empty list to keep track of our testing session logs
session_logs_2 = []

In [14]:
# JavaScript helper to handle the browser webcam stream
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       stream.getVideoTracks()[0].stop();
       video.remove();
       div.remove();
       video = null;
       div = null;
    }

    function createDom() {
       if (div !== null) return;

       div = document.createElement('div');
       div.style.border = '2px solid black';
       div.style.padding = '10px';
       div.style.width = '480px';
       div.style.borderRadius = '10px';
       div.style.background = '#f0f0f0';

       labelElement = document.createElement('div');
       labelElement.style.fontSize = '16px';
       labelElement.style.fontWeight = 'bold';
       labelElement.style.marginBottom = '10px';
       labelElement.innerText = "Camera Feed Initializing...";
       div.appendChild(labelElement);

       video = document.createElement('video');
       video.style.display = 'block';
       video.width = 440;
       video.height = 330;
       video.autoplay = true;
       div.appendChild(video);

       imgElement = document.createElement('img');
       imgElement.style.position = 'absolute';
       imgElement.style.top = '0px';
       imgElement.style.left = '0px';

       var button = document.createElement('button');
       button.innerText = "Stop Testing Feed";
       button.style.marginTop = '10px';
       button.style.padding = '5px 10px';
       button.style.cursor = 'pointer';
       button.onclick = function() {
          shutdown = true;
          if (pendingResolve) {
             pendingResolve({img: null, shutdown: true});
          }
       };
       div.appendChild(button);

       document.body.appendChild(div);
    }

    async function initCamera() {
       createDom();
       stream = await navigator.mediaDevices.getUserMedia({video: true});
       video.srcObject = stream;
       await video.play();

       captureCanvas = document.createElement('canvas');
       captureCanvas.width = video.videoWidth;
       captureCanvas.height = video.videoHeight;
    }

    async function captureFrame(statusText) {
       if (shutdown) {
          removeDom();
          return {img: null, shutdown: true};
       }

       labelElement.innerText = statusText;

       // Wait a brief moment to ensure frame changes
       await new Promise(resolve => setTimeout(resolve, 100));

       var context = captureCanvas.getContext('2d');
       context.drawImage(video, 0, 0, video.videoWidth, video.videoHeight);
       var imgData = captureCanvas.toDataURL('image/jpeg', 0.8);

       return {img: imgData, shutdown: false};
    }

    window.initCamera = initCamera;
    window.captureFrame = captureFrame;
    ''')
  display(js)

# Helper function to convert JavaScript base64 image back to a OpenCV/NumPy BGR image
def js_to_image(js_reply):
  image_bytes = b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=cv2.IMREAD_COLOR)
  return img

In [19]:
import ipywidgets as widgets
import time

# Configurable delay (in seconds) to give you time to switch positions
PAUSE_DURATION = 4

# Dropdown to select what posture you are simulating right now
simulated_state_widget = widgets.Dropdown(
    options=['no_hands', 'one_hand', 'both_hands'],
    value='both_hands',
    description='Simulating:',
)
display(simulated_state_widget)

# Start the web camera
video_stream()
eval_js('initCamera()')

print(f"Camera started. You have a {PAUSE_DURATION}-second delay between each snapshot to adjust your posture.")
print("Click 'Stop Testing Feed' below the video to finish the session.")

# Reset local log for this specific run
run_logs = []

while True:
    current_target = simulated_state_widget.value
    status_text = f"Simulating: {current_target.upper()} | Capturing frame now..."

    # 1. Capture a frame from the JavaScript side
    js_reply = eval_js(f'captureFrame("{status_text}")')
    if js_reply is None or js_reply.get('shutdown', False):
        print("\nTesting session ended by user.")
        break

    # Process image matrix
    img = js_to_image(js_reply['img'])

    # Preprocess matching your dataset structure
    rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    resized_img = cv2.resize(rgb_img, IMG_SIZE)
    input_tensor = np.expand_dims(resized_img, axis=0).astype(np.float32)

    # Core inference execution
    predictions = model.predict(input_tensor, verbose=0)
    score = tf.nn.softmax(predictions[0])
    class_id = np.argmax(score)
    pred_label = CLASS_NAMES[class_id]
    confidence = float(np.max(score))

    # Save log details
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    is_correct = (pred_label == current_target)

    record = {
        "Timestamp": timestamp,
        "Simulated_Ground_Truth": current_target,
        "Model_Prediction": pred_label,
        "Confidence": confidence,
        "Is_Correct": is_correct
    }
    run_logs.append(record)
    session_logs.append(record)

    # Print real-time diagnostic output
    print(f"[{timestamp}] Expected: {current_target:10} | Predicted: {pred_label:10} ({confidence*100:6.2f}%) -> Match: {is_correct}")

    # 2. Countdown sequence to give you a margin to adjust
    print("Preparing for next snapshot... ", end="")
    for i in range(PAUSE_DURATION, 0, -1):
        # Update the video banner dynamically with the remaining countdown time
        countdown_text = f"HOLD POSTURE! Next frame in {i} seconds..."
        eval_js(f'captureFrame("{countdown_text}")')

        print(f"{i}.. ", end="", flush=True)
        time.sleep(1)
    print("Capture!\n" + "-"*60)

Dropdown(description='Simulating:', index=2, options=('no_hands', 'one_hand', 'both_hands'), value='both_hands…

<IPython.core.display.Javascript object>

Camera started. You have a 4-second delay between each snapshot to adjust your posture.
Click 'Stop Testing Feed' below the video to finish the session.
[2026-07-05 11:02:45] Expected: both_hands | Predicted: one_hand   ( 40.97%) -> Match: False
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 11:02:50] Expected: both_hands | Predicted: both_hands ( 38.46%) -> Match: True
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 11:02:56] Expected: both_hands | Predicted: both_hands ( 55.29%) -> Match: True
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-05 11:03:01] Expected: both_hands | Predicted: one_hand   ( 40.57%) -> Match: False
Preparing for next snapshot... 4.. 3.. 2.. 1.. Capture!
------------------------------------------------------------
[2026-07-